In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# 1. Load Dataset
df = pd.read_csv('heart_disease.csv')

# Drop multi-class raw target 'num' to prevent data leakage
X = df.drop(columns=['num', 'target_binary'])
y = df['target_binary']

# 2. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save test dataset for Streamlit app upload requirement
test_df = pd.DataFrame(X_test, columns=X.columns)
test_df['target'] = y_test.values
test_df.to_csv('test_data.csv', index=False)

# Directory for model binaries
os.makedirs('model', exist_ok=True)
joblib.dump(scaler, 'model/scaler.pkl')

# 4. Instantiate 5 Models
models = {
    "Logistic Regression": (LogisticRegression(random_state=42), True),
    "Decision Tree": (DecisionTreeClassifier(random_state=42), False),
    "K-Nearest Neighbor": (KNeighborsClassifier(n_neighbors=5), True),
    "Naive Bayes": (GaussianNB(), True),
    "Random Forest (Ensemble)": (RandomForestClassifier(n_estimators=100, random_state=42), False)
}

metrics_list = []

# 5. Model Training & Evaluation Loop
for name, (model, use_scaled) in models.items():
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    metrics_list.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

    # Save model artifact
    filename = f"model/{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.pkl"
    joblib.dump(model, filename)

# Print Summary Table
metrics_df = pd.DataFrame(metrics_list)
print("=== EVALUATION METRICS SUMMARY ===")
print(metrics_df.to_string(index=False))

=== EVALUATION METRICS SUMMARY ===
           ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
     Logistic Regression    0.8585 0.9251     0.8571  0.8298 0.8432 0.7147
           Decision Tree    0.7415 0.7368     0.7356  0.6809 0.7072 0.4775
      K-Nearest Neighbor    0.8439 0.9020     0.8690  0.7766 0.8202 0.6864
             Naive Bayes    0.8439 0.9131     0.8298  0.8298 0.8298 0.6856
Random Forest (Ensemble)    0.8488 0.9325     0.8316  0.8404 0.8360 0.6957
